# HH Goa 2026 — Task 3
## Face Identification & Blockchain Verification

**Pipeline:** Face scan → genuine reverse-image/web search → candidate face matching → SHA-256 fingerprint → EVM blockchain registration → verification.

> Face matching is a content-matching signal, not proof of real-world identity.


## 1. Project Overview
This notebook is the Colab entry point. The reusable implementation lives under `app/`.

In [ ]:
# If you uploaded the repository ZIP to Colab, unzip it first.
# Otherwise, clone your GitHub repository and cd into it.
# Example:
# !git clone <YOUR_REPO_URL>
# %cd hh-goa-face-blockchain

from pathlib import Path
PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)


## 2. Install Dependencies

In [ ]:
%pip install -q -r requirements.txt
print("Dependencies installed.")


## 3. Configuration / Secrets
Add `SERPAPI_KEY`, `RPC_URL`, `PRIVATE_KEY`, and `CONTRACT_ADDRESS` in Colab Secrets. Never paste secrets into notebook cells or commit them.

In [ ]:
import os

def colab_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

for name in ["SERPAPI_KEY", "RPC_URL", "PRIVATE_KEY", "CONTRACT_ADDRESS"]:
    print(name, "✓ configured" if colab_secret(name) else "✗ missing")


## 4. Face Detection & Embedding

In [ ]:
from app.face import FaceEngine, select_primary_face
import cv2
import matplotlib.pyplot as plt

engine = FaceEngine()
print("InsightFace model loaded.")

def show_faces(image, faces):
    display_img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    for f in faces:
        x1, y1, x2, y2 = f.bbox
        cv2.rectangle(display_img, (x1, y1), (x2, y2), (255, 0, 0), 2)
    plt.figure(figsize=(8, 6))
    plt.imshow(display_img)
    plt.axis("off")
    plt.show()


## 5. Genuine Web/Social Search

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = next(iter(uploaded))
print("Uploaded:", image_path)

image, faces = engine.detect_from_path(image_path)
print(f"Faces detected: {len(faces)}")
if not faces:
    raise ValueError("No face detected. Upload an image containing a clear face.")
show_faces(image, faces)

target_face = select_primary_face(faces)
print("Embedding generated:", target_face.embedding.shape)


In [ ]:
from app.search import search_web_for_image

candidates, raw_search = search_web_for_image(image_path, max_candidates=12)
print(f"Actual search returned {len(candidates)} candidate result(s).")

for i, c in enumerate(candidates, 1):
    print(f"\nCandidate {i}")
    print("Platform:", c.platform)
    print("URL:", c.url)
    print("Image URL:", c.image_url)
    print("Title:", c.title)


## 6. Candidate Matching

In [ ]:
from app.matcher import rank_candidates, MatchConfig

MATCH_THRESHOLD = 0.55  # visible/configurable demo parameter
ranked = rank_candidates(
    engine,
    target_face.embedding,
    candidates,
    config=MatchConfig(threshold=MATCH_THRESHOLD),
)

print(f"Configured face-match threshold: {MATCH_THRESHOLD}")
print("\nSEARCH RESULTS")
for i, c in enumerate(ranked, 1):
    score = "N/A" if c.face_similarity is None else f"{c.face_similarity * 100:.1f}%"
    print(f"\nCandidate {i}")
    print("Platform:", c.platform)
    print("URL:", c.url)
    print("Face similarity:", score)
    print("Status:", c.status)

matches = [c for c in ranked if c.status == "MATCH" and c.downloaded_image]
if not matches:
    raise RuntimeError("No valid face match found. Try another image or adjust the visible threshold.")
best = matches[0]
print("\nBEST MATCH")
print(best.public_dict())


## 7. Content Fingerprinting

In [ ]:
from app.hashing import canonical_content_record, content_fingerprint

content_hash = content_fingerprint(
    image_bytes=best.downloaded_image,
    source_url=best.url,
    title=best.title,
    platform=best.platform or "",
    snippet=best.snippet or "",
)

print("CONTENT FINGERPRINT")
print("SHA-256:", content_hash)
print("✓ Fingerprint generated")

canonical_record = canonical_content_record(
    image_bytes=best.downloaded_image,
    source_url=best.url,
    title=best.title,
    platform=best.platform or "",
    snippet=best.snippet or "",
)
print("Canonical record bytes:", len(canonical_record))


## 8A. Deploy the Solidity Contract (optional first-time setup)

In [ ]:
# OPTIONAL: deploy ContentVerifier to your configured EVM testnet.
# Run once per deployment, then copy the printed address into the CONTRACT_ADDRESS Secret.
#
# !python scripts/deploy_contract.py
#
# The script requires RPC_URL and PRIVATE_KEY in the environment.
# In Colab, you can export secrets for this one process:
#
# import os
# os.environ["RPC_URL"] = colab_secret("RPC_URL")
# os.environ["PRIVATE_KEY"] = colab_secret("PRIVATE_KEY")
# !python scripts/deploy_contract.py


## 8. Blockchain Connection

In [ ]:
import json
from pathlib import Path
from app.blockchain import ContentVerifierClient, load_abi

abi_path = Path("artifacts/ContentVerifier.abi.json")
if not abi_path.exists():
    raise FileNotFoundError(
        "Deploy the contract first or provide artifacts/ContentVerifier.abi.json "
        "and CONTRACT_ADDRESS."
    )

abi = load_abi(abi_path)
RPC_URL = colab_secret("RPC_URL")
PRIVATE_KEY = colab_secret("PRIVATE_KEY")
CONTRACT_ADDRESS = colab_secret("CONTRACT_ADDRESS")

client = ContentVerifierClient(
    rpc_url=RPC_URL,
    private_key=PRIVATE_KEY,
    contract_address=CONTRACT_ADDRESS,
    abi=abi,
)
print("Connected to chain:", client.w3.eth.chain_id)
print("Wallet:", client.account.address)


## 9. Blockchain Registration

In [ ]:
tx_hash = client.register(content_hash, best.url)
print("✓ Blockchain registration complete")
print("Transaction:", tx_hash)


## 10. Blockchain Verification

In [ ]:
verification = client.verify(content_hash)

print("BLOCKCHAIN VERIFICATION")
print("Exists:", verification["exists"])
print("Source URL:", verification["source_url"])
print("Timestamp:", verification["timestamp"])
print("Submitter:", verification["submitter"])
print("Status:", "✓ VERIFIED" if verification["exists"] else "✗ NOT VERIFIED")


## 11. End-to-End Pipeline

In [ ]:
def run_pipeline(image_path, threshold=MATCH_THRESHOLD):
    image, faces = engine.detect_from_path(image_path)
    if not faces:
        raise ValueError("No face detected.")

    target = select_primary_face(faces)
    candidates, raw = search_web_for_image(image_path, max_candidates=12)

    ranked = rank_candidates(
        engine,
        target.embedding,
        candidates,
        config=MatchConfig(threshold=threshold),
    )
    matches = [c for c in ranked if c.status == "MATCH" and c.downloaded_image]
    if not matches:
        raise RuntimeError("No candidate reached the configured face-match threshold.")

    best = matches[0]
    fingerprint = content_fingerprint(
        image_bytes=best.downloaded_image,
        source_url=best.url,
        title=best.title,
        platform=best.platform or "",
        snippet=best.snippet or "",
    )

    tx = client.register(fingerprint, best.url)
    verification = client.verify(fingerprint)

    return {
        "faces_detected": len(faces),
        "best_match": best.public_dict(),
        "content_hash": fingerprint,
        "transaction": tx,
        "verification": verification,
    }

# Run only after all configuration cells above succeed:
# result = run_pipeline(image_path)
# result


## 12. Demo — final output

In [ ]:
# If `result` was produced above:
# print("========================================")
# print("FINAL VERIFICATION")
# print("========================================")
# print("Face Detection          ✓")
# print("Face Match              ✓")
# print("Web/Social Search       ✓")
# print("Matching Post Found     ✓")
# print("SHA-256 Fingerprint     ✓")
# print("Blockchain Registration ✓")
# print("Blockchain Verification ✓")
# print("RESULT:", "VERIFIED" if result["verification"]["exists"] else "NOT VERIFIED")
# print("========================================")


## 13. Optional Gradio Demo UI

In [ ]:
from app.ui import build_gradio_demo

# This UI exposes SEARCH & MATCH, REGISTER ON BLOCKCHAIN, and VERIFY.
# It uses the same core modules as run_pipeline().
#
# Run after the blockchain `client` is configured:
# demo = build_gradio_demo(engine, blockchain_client=client)
# demo.launch(share=True, debug=True)
#
# If blockchain is not configured yet, you can still launch:
# demo = build_gradio_demo(engine, blockchain_client=None)
# demo.launch(share=True, debug=True)
